# Poisson equation on a grid: classical LU vs HHL hybrid

Solve `L u = b` on a 4×4 Dirichlet grid. The same `linear_solve` trace runs on cpu (LU) or cpu+sim (HHL hybrid). The user code is identical — the planner picks the algorithm based on the available targets.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** discrete Poisson equation `∇² u = b` on a real-space grid. **Classical algorithm:** LU / Cholesky / conjugate gradient. **Quantum algorithm:** HHL (Harrow-Hassidim-Lloyd, 2009) — block-encode the Laplacian, QPE, controlled rotation, post-select. Both solve `Ax=b`; the user writes `linear_solve(A, b)` and lets the planner choose.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.core import types
from uniqx.domains.physics.kernels import grid_laplacian
from uniqx.ops import shape
from uniqx.ops.primitives.evolution import apply_linear

nx, ny = 4, 4
n = nx * ny
dx = 1.0 / (nx + 1)
dy = 1.0 / (ny + 1)

@trace
def prog(rhs):
    l_flat = grid_laplacian(nx=nx, ny=ny, nz=1, dx=dx, dy=dy)
    l_mat = shape.reshape(l_flat, result_type=types.tensor("f64", [n, n]), shape=[n, n])
    return apply_linear(l_mat, rhs)

rhs = np.ones(n)
module = prog(rhs.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")


## Compare to a classical oracle

In [ ]:
# Reference: classical numpy.linalg.solve.
def ref_grid_laplacian_2d(nx, ny, dx, dy):
    n = nx * ny
    L = np.zeros((n, n))
    inv_dx2, inv_dy2 = 1.0 / (dx * dx), 1.0 / (dy * dy)
    for j in range(ny):
        for i in range(nx):
            k = j * nx + i
            L[k, k] = -2.0 * (inv_dx2 + inv_dy2)
            if i + 1 < nx:
                L[k, k + 1] += inv_dx2; L[k + 1, k] += inv_dx2
            if j + 1 < ny:
                L[k, k + nx] += inv_dy2; L[k + nx, k] += inv_dy2
    return L

L = ref_grid_laplacian_2d(nx, ny, dx, dy)
print(f"Classical L @ b = {(L @ rhs)[:8]}")
